# Chapter 10: Model Context Protocol (MCP)

**Part:** Part IV: Autonomous Agents, MCP & Production MLOps  
**Chapter Identifier:** `ch10`  

## Core Concepts Covered in this Chapter:
- **What is MCP**
- **N×M Problem**
- **Hosts/Clients/Servers**
- **JSON-RPC**
- **Tools**
- **Resources**
- **Prompts**
- **Sampling**
- **Capability Negotiation**
- **stdio vs. Streamable HTTP**
- **Lifecycle**
- **Tool Schemas**
- **OAuth**
- **Building MCP Servers**
- **Connecting IDEs/Apps**
- **Local vs. Remote**
- **Prompt Injection**
- **Permissions**
- **MCP Inspector**

---


## 1. What is MCP & Solving the N×M Integration Problem

Before MCP, connecting N AI applications (Claude Desktop, Antigravity, Cursor, LangChain) to M data sources (GitHub, Slack, Postgres, local files) required N×M bespoke custom plugins. MCP standardizes this into an open JSON-RPC 2.0 protocol: any Host client can connect to any MCP Server.

```mermaid
graph TD
    subgraph "Before MCP: N × M Chaos"
        H1[App 1] --> S1[Tool 1]
        H1 --> S2[Tool 2]
        H2[App 2] --> S1
        H2 --> S2
    end
    subgraph "With MCP: Universal Standard"
        HostA[Host Client 1] --> Proto["MCP Protocol (JSON-RPC 2.0)"]
        HostB[Host Client 2] --> Proto
        Proto --> Srv1["MCP Server (Postgres)"]
        Proto --> Srv2["MCP Server (GitHub)"]
        Proto --> Srv3["MCP Server (Local FS)"]
    end
```


In [ ]:
print("MCP solves the N x M integration problem through universal JSON-RPC 2.0 primitives.")

## 2. The Three Core Primitives: Tools, Resources, and Prompts

MCP defines three standard capabilities: (1) Tools — executable functions with JSON schemas that models can invoke; (2) Resources — file-like data and schemas that clients can read; (3) Prompts — pre-defined prompt templates provided by servers.

```mermaid
classDiagram
    class MCPServer {
        +tools_list() List[Tool]
        +tools_call(name, args) CallResult
        +resources_list() List[Resource]
        +resources_read(uri) ResourceContent
        +prompts_list() List[Prompt]
        +prompts_get(name, args) PromptResult
    }
    class Tool {
        +str name
        +str description
        +dict inputSchema
    }
    class Resource {
        +str uri
        +str name
        +str mimeType
    }
    MCPServer --> Tool
    MCPServer --> Resource
```


In [ ]:
import json

def build_mcp_request(method: str, params: dict, request_id: int = 1) -> str:
    return json.dumps({
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": params
    })

def build_mcp_response(result: dict, request_id: int = 1) -> str:
    return json.dumps({
        "jsonrpc": "2.0",
        "id": request_id,
        "result": result
    })

req = build_mcp_request("tools/call", {"name": "fetch_schema", "arguments": {"table": "users"}})
print("JSON-RPC MCP Request:\n", req)

## 3. MCP Transports: stdio vs. Streamable HTTP (SSE)

MCP supports two primary transport mechanisms: (1) stdio — standard input/output pipes for fast, secure local process communication without opening network ports; (2) Streamable HTTP / Server-Sent Events (SSE) for remote enterprise servers.


In [ ]:
transports = {
    "stdio": {
        "use_case": "Local desktop apps, CLI tools, local files",
        "security": "Inherits operating system user permissions, no open ports"
    },
    "Streamable HTTP (SSE)": {
        "use_case": "Remote enterprise cloud services, multi-tenant servers",
        "security": "TLS, OAuth 2.0 Bearer tokens, CORS validation"
    }
}
for name, spec in transports.items():
    print(f"[{name.upper()}]\n  Best For: {spec['use_case']}\n  Security: {spec['security']}\n")

---
## Hands-On Capstone Lab: Build a Fully Compliant In-Memory MCP Server with Tool & Resource Handling

### Objective:
Build an MCP Server class supporting `tools/list`, `tools/call`, and `resources/read` JSON-RPC dispatching with input validation.

### Instructions:
1. Implement JSON-RPC router handling `tools/list` and `tools/call`.
2. Expose a tool `get_system_metrics`.
3. Execute and test with valid and invalid calls.


In [ ]:
class LightweightMCPServer:
    def __init__(self, server_name: str, version: str):
        self.server_name = server_name
        self.version = version
        self.tools = {
            "get_system_metrics": {
                "description": "Returns simulated CPU, memory, and disk utilization metrics.",
                "inputSchema": {"type": "object", "properties": {"detailed": {"type": "boolean"}}},
                "handler": lambda detailed=False: {"cpu_pct": 24.5, "mem_pct": 58.2, "disk_gb_free": 124.0}
            }
        }

    def handle_jsonrpc(self, raw_message: str) -> str:
        req = json.loads(raw_message)
        method = req.get("method")
        msg_id = req.get("id", 1)
        
        if method == "initialize":
            return build_mcp_response({
                "protocolVersion": "2024-11-05",
                "serverInfo": {"name": self.server_name, "version": self.version},
                "capabilities": {"tools": {}, "resources": {}}
            }, msg_id)
            
        elif method == "tools/list":
            tool_list = [{"name": name, "description": t["description"], "inputSchema": t["inputSchema"]} for name, t in self.tools.items()]
            return build_mcp_response({"tools": tool_list}, msg_id)
            
        elif method == "tools/call":
            params = req.get("params", {})
            name = params.get("name")
            args = params.get("arguments", {})
            if name not in self.tools:
                return json.dumps({"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32601, "message": f"Tool '{name}' not found"}})
            result = self.tools[name]["handler"](**args)
            return build_mcp_response({"content": [{"type": "text", "text": json.dumps(result)}]}, msg_id)
            
        return json.dumps({"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32600, "message": "Invalid Request"}})

server = LightweightMCPServer("dev-ops-mcp", "1.0.0")
init_resp = server.handle_jsonrpc(build_mcp_request("initialize", {}))
print("Handshake Response:", init_resp)

call_resp = server.handle_jsonrpc(build_mcp_request("tools/call", {"name": "get_system_metrics", "arguments": {"detailed": True}}))
print("Tool Call Response:", call_resp)

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary architectural problem that the Model Context Protocol (MCP) eliminates?
- **A)** The N×M fragmentation problem where every client required custom connectors for every data source
- **B)** The cost of buying GPU hardware
- **C)** The need to write unit tests
- **D)** The requirement to use Python

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) The N×M fragmentation problem where every client required custom connectors for every data source

**Explanation:** MCP standardizes client-server protocol over JSON-RPC, transforming N×M custom integrations into N clients and M servers communicating through one standard.
</details>

### Question 2: Which protocol standard forms the underlying communication format of MCP messages?
- **A)** GraphQL
- **B)** JSON-RPC 2.0
- **C)** SOAP XML
- **D)** Protocol Buffers v2

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) JSON-RPC 2.0

**Explanation:** MCP strictly leverages JSON-RPC 2.0 specifying requests, responses, notifications, error codes, and unique message IDs.
</details>

### Question 3: What are the three core server primitives exposed by an MCP Server?
- **A)** Headers, Cookies, and Sessions
- **B)** Tools, Resources, and Prompts
- **C)** Tables, Views, and Triggers
- **D)** Models, Weights, and Checkpoints

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) Tools, Resources, and Prompts

**Explanation:** MCP servers expose Tools (executable actions), Resources (context data/content), and Prompts (reusable parameterized prompt templates).
</details>

### Question 4: Why is the `stdio` transport advantageous for local desktop AI tools (like Claude Desktop or Antigravity)?
- **A)** It transmits data faster than light
- **B)** It runs through standard OS process pipes without exposing local network ports or introducing remote network attack vectors
- **C)** It requires no CPU power
- **D)** It only works with cloud databases

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) It runs through standard OS process pipes without exposing local network ports or introducing remote network attack vectors

**Explanation:** stdio processes are launched as child processes communicating over stdin/stdout, isolating them behind host operating system permissions without listening sockets.
</details>
